In [1]:
import pandas as pd
import glob
import os

print("--- Starting Master Panel Data Construction (2007 - 2024) ---")

# 1. กำหนดรายชื่อคอลัมน์มาตรฐานที่เราต้องการเก็บไว้ใน Panel Data 
# เพื่อคัดกรองตัวแปรขยะของแต่ละระลอกออกไป ไม่ให้ไฟล์บวม
core_panel_columns = [
    'interview__key', 'hhid', 'survey_year', 'shocks_Group',
    'impact_3way', 'impact_2way', 'recovery_std', 'cons_label',
    'coping_savings', 'coping_insurance', 'coping_informal_borrow', 
    'coping_formal_borrow', 'coping_assets', 'coping_gov_help'
]

all_wave_data = []

# 2. ค้นหาไฟล์ CSV ที่คลีนเสร็จแล้วทั้งหมดในเครื่อง
# หมายเหตุ: มั่นใจว่าชื่อไฟล์คลีนของคุณระบุปีตรงตามตรรกะนี้ เช่น shocks_2007_cleaned.csv
csv_files = glob.glob('shocks_*_cleaned.csv')

for file_path in sorted(csv_files):
    print(f"Processing and extracting core panel variables from: {file_path}")
    df_wave = pd.read_csv(file_path)
    
    # ตรวจเช็คกรณีชื่อคอลัมน์ Coping ในยุคเก่าที่อาจไม่ได้ถูกปรับระหว่างทาง ให้สอดคล้องกันก่อน Append
    rename_dict = {
        'coping_used_savings': 'coping_savings',
        'coping_used_insurance': 'coping_insurance',
        'coping_borrowed_informal': 'coping_informal_borrow',
        'coping_borrowed_formal': 'coping_formal_borrow',
        'coping_sold_assets': 'coping_assets',
        'consumption_label': 'cons_label'
    }
    df_wave = df_wave.rename(columns=rename_dict)
    
    # หากระลอกใดไม่มีคอลัมน์ระบุ hhid ดั้งเดิม ให้พยายามเก็บ ID พื้นฐานไว้สำหรับการระบุครัวเรือน
    if 'hhid' not in df_wave.columns and 'interview__key' in df_wave.columns:
        df_wave['hhid'] = df_wave['interview__key']
        
    # คัดเฉพาะคอลัมน์ที่มีอยู่ในข้อมูลจริงและอยู่ในกลุ่มแกนหลัก
    existing_cols = [col for col in core_panel_columns if col in df_wave.columns]
    df_filtered = df_wave[existing_cols]
    
    all_wave_data.append(df_filtered)

# 3. รวมร่างข้อมูลในแนวตั้งข้ามช่วงเวลา (Longitudinal Form)
master_panel_df = pd.concat(all_wave_data, axis=0, ignore_index=True)

# เติมคอลัมน์ที่ขาดหายในบาง Wave ให้เป็น NaN เพื่อรักษาโครงสร้างตาราง
for col in core_panel_columns:
    if col not in master_panel_df.columns:
        master_panel_df[col] = np.nan

# จัดเรียงคอลัมน์ให้สวยงามและเป็นระเบียบ
master_panel_df = master_panel_df[core_panel_columns]

print(f"\n✨ Master Panel Construction Complete! ✨")
print(f"Total Longitudinal Observations: {len(master_panel_df)} shock events across time.")
print(f"Data Breakdown by Survey Year:")
print(master_panel_df['survey_year'].value_counts().sort_index())

# 4. ส่งออกไฟล์แผงข้อมูลที่สมบูรณ์ที่สุด
output_master_file = 'master_panel_shocks_2007_2024.csv'
master_panel_df.to_csv(output_master_file, index=False)
print(f"\n💾 Saved master file successfully as: {output_master_file}")

--- Starting Master Panel Data Construction (2007 - 2024) ---
Processing and extracting core panel variables from: shocks_2007_cleaned.csv
Processing and extracting core panel variables from: shocks_2008_cleaned.csv
Processing and extracting core panel variables from: shocks_2010_cleaned.csv
Processing and extracting core panel variables from: shocks_2011_cleaned.csv
Processing and extracting core panel variables from: shocks_2013_cleaned.csv
Processing and extracting core panel variables from: shocks_2016_cleaned.csv
Processing and extracting core panel variables from: shocks_2017_cleaned.csv
Processing and extracting core panel variables from: shocks_2019_cleaned.csv
Processing and extracting core panel variables from: shocks_2022_cleaned.csv
Processing and extracting core panel variables from: shocks_2024_cleaned.csv

✨ Master Panel Construction Complete! ✨
Total Longitudinal Observations: 23057 shock events across time.
Data Breakdown by Survey Year:
survey_year
2007.0    2020
2008

/var/folders/7l/zffgp8nd1wbb6qkk7lg__0_c0000gn/T/ipykernel_9704/3162969437.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_wave['hhid'] = df_wave['interview__key']
